In [1]:
#Importations et Chargement Initial
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Chargement des données
df = pd.read_csv('C:/Users/dell/Desktop/projet-datamining/Dataminig-Project/data/raw/Road Accident Data.csv')  
print(f"Dataset chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.dtypes)
print(df.head())


Dataset chargé: 307973 lignes, 21 colonnes
Accident_Index                 object
Accident Date                  object
Day_of_Week                    object
Junction_Control               object
Junction_Detail                object
Accident_Severity              object
Latitude                      float64
Light_Conditions               object
Local_Authority_(District)     object
Carriageway_Hazards            object
Longitude                     float64
Number_of_Casualties            int64
Number_of_Vehicles              int64
Police_Force                   object
Road_Surface_Conditions        object
Road_Type                      object
Speed_limit                     int64
Time                           object
Urban_or_Rural_Area            object
Weather_Conditions             object
Vehicle_Type                   object
dtype: object
  Accident_Index Accident Date Day_of_Week          Junction_Control  \
0  200901BS70001      1/1/2021    Thursday  Give way or uncontrolled   
1

In [2]:
#Nettoyage des Données (Cleaning)
# 2.1 Suppression des doublons
df.drop_duplicates(subset=['Accident_Index'], inplace=True)
df.set_index('Accident_Index', inplace=True)
print(f"Doublons supprimés: {df.shape}")

# 2.2 Gestion des valeurs manquantes
print("\nValeurs manquantes par colonne:")
print(df.isnull().sum())

# Imputation numérique (moyenne pour variables continues)
num_cols = ['Latitude', 'Longitude', 'Speed_limit', 'Number_of_Casualties', 'Number_of_Vehicles']
imputer_num = SimpleImputer(strategy='mean')
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# Imputation catégorielle (mode)
cat_cols = ['Junction_Control', 'Road_Surface_Conditions', 'Light_Conditions', 
           'Weather_Conditions', 'Road_Type', 'Urban_or_Rural_Area']
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])
#suppression de la colonne carriageway_hazards
df.drop(columns=['Carriageway_Hazards'], inplace=True)
print("Valeurs manquantes traitées")


Doublons supprimés: (197644, 20)

Valeurs manquantes par colonne:
Accident Date                      0
Day_of_Week                        0
Junction_Control                   0
Junction_Detail                    0
Accident_Severity                  0
Latitude                           0
Light_Conditions                   0
Local_Authority_(District)         0
Carriageway_Hazards           194474
Longitude                          0
Number_of_Casualties               0
Number_of_Vehicles                 0
Police_Force                       0
Road_Surface_Conditions          304
Road_Type                       1013
Speed_limit                        0
Time                               0
Urban_or_Rural_Area                0
Weather_Conditions              3537
Vehicle_Type                       0
dtype: int64
Valeurs manquantes traitées


In [3]:
#Traitement des Outliers
# 3.1 Détection et limitation des outliers (méthode IQR)
def clip_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower_bound, upper_bound)
    return df

# Application sur variables numériques critiques
for col in ['Speed_limit', 'Number_of_Casualties', 'Number_of_Vehicles']:
    df = clip_outliers(df, col)

print("Outliers traités (Speed_limit, Casualties, Vehicles)")


Outliers traités (Speed_limit, Casualties, Vehicles)


In [4]:
# 4.1 Conversion des dates et temps
df['Accident Date'] = pd.to_datetime(df['Accident Date'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce')

df['Year'] = df['Accident Date'].dt.year
df['Month'] = df['Accident Date'].dt.month
df['Hour'] = df['Time'].dt.hour

# 4.2 Correction des types
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df['Speed_limit'] = pd.to_numeric(df['Speed_limit'], errors='coerce')

# 4.3 Feature Engineering
df['Is_Night'] = (df['Hour'] < 6) | (df['Hour'] >= 20)
df['Weekend'] = df['Day_of_Week'].isin(['Saturday', 'Sunday'])

In [5]:
#Encodage Catégoriel (One-Hot + Label)
# 5.1 Variables catégorielles à encoder
high_cardinality = ['Local_Authority_(District)', 'Police_Force', 'Vehicle_Type']
low_cardinality = ['Weather_Conditions', 'Road_Type', 'Urban_or_Rural_Area']

# Label Encoding pour haute cardinalité
le = LabelEncoder()
for col in high_cardinality:
    df[f'{col}_encoded'] = le.fit_transform(df[col].astype(str))

# One-Hot Encoding pour basse cardinalité
df = pd.get_dummies(df, columns=low_cardinality, prefix=low_cardinality)
#gérer les autres en Label Encoding
other_cats = ['Junction_Control', 'Junction_Detail', 'Road_Surface_Conditions', 'Light_Conditions']

for col in other_cats:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    
#Encodage de la variable cible
df['Accident_Severity_encoded'] = LabelEncoder().fit_transform(df['Accident_Severity'])    
print("Encodage terminé")
print("Colonnes après encodage:", df.shape[1])


Encodage terminé
Colonnes après encodage: 40


In [6]:
# Suppression des colonnes inutiles pour ML
df.drop(columns=['Accident_Severity', 'Accident Date', 'Time',
                 'Local_Authority_(District)', 'Police_Force', 'Vehicle_Type'], inplace=True)

print("Colonnes après suppression des redondantes/inutiles:", df.shape[1])

Colonnes après suppression des redondantes/inutiles: 34


In [7]:
# Encodage Label des colonnes catégorielles restantes
from sklearn.preprocessing import LabelEncoder

cols_to_encode = ['Day_of_Week', 'Junction_Control', 'Junction_Detail', 
                  'Light_Conditions', 'Road_Surface_Conditions']

for col in cols_to_encode:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

print("Encodage Label terminé pour les colonnes restantes")

Encodage Label terminé pour les colonnes restantes


In [8]:
#Normalisation/Scaling
# 6.0 Pré-validation des données brutes (avant standardisation)
df_validation = df.copy()

# 6.1 Scaling des variables numériques
scale_cols = ['Latitude', 'Longitude', 'Speed_limit', 'Number_of_Casualties', 
              'Number_of_Vehicles', 'Year', 'Month', 'Hour']
scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print("Données normalisées")


Données normalisées


In [9]:
#Validation de Cohérence (règles métier)
# 7.1 Vérifications logiques
def validate_data(df):
    issues = []
    
    # Casualties <= Vehicles * 4 (max 4 par véhicule)
    if (df['Number_of_Casualties'] > df['Number_of_Vehicles'] * 4).any():
        issues.append("Incohérence: Casualties > Vehicles*4")
    
    # Speed limits réalistes (20-70 mph)
    if not df['Speed_limit'].between(20, 70).all():
        issues.append("Speed_limit hors plage réaliste")
    
    # Coordonnées UK plausibles
    if not (df['Latitude'].between(49, 61).all() and df['Longitude'].between(-8, 2).all()):
        issues.append("Coordonnées hors UK")
    
    return issues

# Utiliser le dataframe non normalisé pour les règles métier
issues = validate_data(df_validation)
if issues:
    print("⚠️  PROBLÈMES DÉTECTÉS:", issues)
else:
    print("✅ Toutes les validations passées!")

⚠️  PROBLÈMES DÉTECTÉS: ['Speed_limit hors plage réaliste']


In [10]:
# 8.1 Sauvegarde en formats multiples
import os
os.makedirs('../data/processed', exist_ok=True)  # Création du dossier si nécessaire

csv_path = '../data/processed/accidents_cleaned.csv'
parquet_path = '../data/processed/accidents_cleaned.parquet'

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

# 8.2 Rapport final
rapport = {
    'lignes_finales': df.shape[0],
    'colonnes_finales': df.shape[1],
    'memoire_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    'valeurs_manquantes': df.isnull().sum().sum(),
    'dtypes': df.dtypes.value_counts().to_dict()
}

print("\n" + "="*50)
print("PHASE DE PRÉPARATION TERMINÉE!")
print("="*50)
for k, v in rapport.items():
    print(f"{k}: {v}")
print("\nFichiers sauvegardés:")
print(f"- {csv_path}")
print(f"- {parquet_path}")

if os.path.exists(csv_path) and os.path.exists(parquet_path):
    print("✅ Vérification: fichiers créés avec succès dans data/processed/")
else:
    print("❌ Erreur: un ou plusieurs fichiers sont manquants dans data/processed/")

print("✅ Prêt pour MLflow et modélisation!")



PHASE DE PRÉPARATION TERMINÉE!
lignes_finales: 197644
colonnes_finales: 34
memoire_mb: 44.56
valeurs_manquantes: 0
dtypes: {dtype('bool'): 17, dtype('int64'): 9, dtype('float64'): 8}

Fichiers sauvegardés:
- ../data/processed/accidents_cleaned.csv
- ../data/processed/accidents_cleaned.parquet
✅ Vérification: fichiers créés avec succès dans data/processed/
✅ Prêt pour MLflow et modélisation!


In [11]:
print("\nValeurs manquantes par colonne:")
print(df.isnull().sum())


Valeurs manquantes par colonne:
Accident_Index                              0
Day_of_Week                                 0
Junction_Control                            0
Junction_Detail                             0
Latitude                                    0
Light_Conditions                            0
Longitude                                   0
Number_of_Casualties                        0
Number_of_Vehicles                          0
Road_Surface_Conditions                     0
Speed_limit                                 0
Year                                        0
Month                                       0
Hour                                        0
Is_Night                                    0
Weekend                                     0
Local_Authority_(District)_encoded          0
Police_Force_encoded                        0
Vehicle_Type_encoded                        0
Weather_Conditions_Fine + high winds        0
Weather_Conditions_Fine no high winds       0
W

In [12]:
df.columns

Index(['Accident_Index', 'Day_of_Week', 'Junction_Control', 'Junction_Detail',
       'Latitude', 'Light_Conditions', 'Longitude', 'Number_of_Casualties',
       'Number_of_Vehicles', 'Road_Surface_Conditions', 'Speed_limit', 'Year',
       'Month', 'Hour', 'Is_Night', 'Weekend',
       'Local_Authority_(District)_encoded', 'Police_Force_encoded',
       'Vehicle_Type_encoded', 'Weather_Conditions_Fine + high winds',
       'Weather_Conditions_Fine no high winds',
       'Weather_Conditions_Fog or mist', 'Weather_Conditions_Other',
       'Weather_Conditions_Raining + high winds',
       'Weather_Conditions_Raining no high winds',
       'Weather_Conditions_Snowing + high winds',
       'Weather_Conditions_Snowing no high winds',
       'Road_Type_Dual carriageway', 'Road_Type_One way street',
       'Road_Type_Roundabout', 'Road_Type_Single carriageway',
       'Road_Type_Slip road', 'Urban_or_Rural_Area_Rural',
       'Urban_or_Rural_Area_Urban', 'Accident_Severity_encoded'],
      d

In [13]:
df["Vehicle_Type_encoded"]

0          2
1         13
2         13
3          9
4          2
          ..
307968     2
307969     2
307970     2
307971     9
307972     2
Name: Vehicle_Type_encoded, Length: 197644, dtype: int64